In [1]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
import sys
import os

try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(notebook_dir, '..')))
from pcp_project.pipeline import SubjectPipeline

class BandPassFilter(BaseEstimator, TransformerMixin):
    def __init__(self, frequency_bands):
        self.frequency_bands = frequency_bands
    def fit(self, X, y=None, **kwargs):
        return self
    def transform(self, X, y=None):
        return X

class StateSelector(BaseEstimator, TransformerMixin):
    def __init__(self, states):
        self.states = states
    def fit(self, X, y=None, **kwargs):
        return self
    def fit_transform(self, X, y=None, groups=None):
        return X, groups
    def transform(self, X, y=None, groups=None):
        return X

class SlidingWindow(BaseEstimator, TransformerMixin):
    def __init__(self, length, step_size):
        self.length = length
        self.step_size = step_size
    def fit(self, X, y=None, **kwargs):
        return self
    def transform(self, X, y=None):
        if isinstance(X, list) or (isinstance(X, np.ndarray) and X.dtype == object):
            return np.array([np.expand_dims(subj_x.T, axis=0) for subj_x in X])
        return np.expand_dims(X.T, axis=0)

class BatchedOAS(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None, **kwargs):
        return self
    def transform(self, X, y=None):
        if isinstance(X, np.ndarray) and X.ndim == 4:
            n_subj, n_epochs, channels, _ = X.shape
            return np.zeros((n_subj, n_epochs, channels, channels))
        n_epochs, channels, _ = X.shape
        return np.zeros((n_epochs, channels, channels))

try:
    from pyriemann.tangentspace import TangentSpace
except ImportError:
    class TangentSpace(BaseEstimator, TransformerMixin):
        def fit(self, X, y=None, **kwargs):
            return self
        def transform(self, X, y=None):
            if X.ndim == 4:
                n_subj, n_epochs, channels, _ = X.shape
                features = channels * (channels + 1) // 2
                return np.zeros((n_subj, n_epochs, features))
            n_epochs, channels, _ = X.shape
            features = channels * (channels + 1) // 2
            return np.zeros((n_epochs, features))

class FeatureConsolidator(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None, **kwargs):
        return self
    def transform(self, X, y=None):
        if X.ndim == 3:
            return np.mean(X, axis=1)
        return np.mean(X, axis=0, keepdims=True)


In [ ]:
import csv

labels_path = os.path.join(notebook_dir, '..', 'data', 'labels_reduced.csv')

target_label = 0.0
with open(labels_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['EEG_ID'] == 'AD_002':
            target_label = float(row['SCID5_CV_OCD'])
            break

data_path = os.path.join(notebook_dir, '..', 'data', 'AD_002.npz')
data = np.load(data_path)
X_subject = data['X']
y_state = data['y'][0] if data['y'].ndim > 1 else data['y']

dataset = np.array([X_subject, X_subject, X_subject, X_subject], dtype=object)
labels = np.array([target_label, 1.0 - target_label, target_label, 1.0 - target_label])

mask_eyes_open = (y_state == 0)
subject_masks = np.array([mask_eyes_open, mask_eyes_open, mask_eyes_open, mask_eyes_open], dtype=object)

pipe = SubjectPipeline(
    steps=[
        ("filter", BandPassFilter(frequency_bands=[[5, 10], [13, 35]])),
        ("select", StateSelector(states=["eyes_open"])),
        ("epoch", SlidingWindow(length=200, step_size=50)),
        ("covariance", BatchedOAS()),
        ("features", TangentSpace()), 
        ("consolidator", FeatureConsolidator()),
        ("clf", LogisticRegression()), 
    ],
    mask=subject_masks
)

scores1 = cross_validate(pipe, X=dataset, y=labels, cv=2)
print("SCID5_CV_OCD (AD_002) Label:", target_label)
print("Option 1 Scores:", scores1)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/ulasayyilmaz/Desktop/PCP/pcp_project-1/examples/../data-20260608/AD_002.npz'